# Row Counts & Sizes – Car Workshop

How much data do I have? Discovers every schema and table in the `car_workshop`
catalog dynamically, then reports per table:

- row count (`count(*)`)
- number of data files and size on disk (`DESCRIBE DETAIL`)

plus per-schema and grand totals, and a **date-coverage check** for the backfill
(missing days detection).

In [ ]:
import pandas as pd

CATALOG = 'car_workshop'
SKIP_SCHEMAS = {'information_schema', 'default'}

schemas = [row[0] for row in spark.sql(f'SHOW SCHEMAS IN {CATALOG}').collect()
           if row[0] not in SKIP_SCHEMAS]
print(f'schemas: {schemas}')

results = []
for schema in schemas:
    for row in spark.sql(f'SHOW TABLES IN {CATALOG}.{schema}').collect():
        full_name = f'{CATALOG}.{schema}.{row.tableName}'
        try:
            rows = spark.table(full_name).count()
            detail = spark.sql(f'DESCRIBE DETAIL {full_name}').collect()[0]
            results.append({
                'schema': schema,
                'table': row.tableName,
                'rows': rows,
                'files': detail['numFiles'],
                'size_mb': round(detail['sizeInBytes'] / 1024 / 1024, 1),
            })
        except Exception as e:
            results.append({'schema': schema, 'table': row.tableName,
                            'rows': None, 'files': None, 'size_mb': None})
            print(f'  ERROR {full_name}: {str(e)[:100]}')

counts = pd.DataFrame(results).sort_values(['schema', 'rows'], ascending=[True, False])
display(counts)

In [ ]:
print(f"TOTAL: {counts['rows'].sum():,.0f} rows, "
      f"{counts['size_mb'].sum() / 1024:.2f} GB, "
      f"{counts['files'].sum():,.0f} files in {len(counts)} tables")
print()
for schema, group in counts.groupby('schema'):
    print(f"  {schema:<8} {group['rows'].sum():>15,.0f} rows "
          f"{group['size_mb'].sum() / 1024:>8.2f} GB {group['files'].sum():>8,.0f} files")

## Backfill date coverage

`fact_employee_schedules` has exactly one batch per generated day, so it works
as a completion marker – any gap in its dates means a missing backfill day.

In [ ]:
coverage = (spark.table(f'{CATALOG}.fact.fact_employee_schedules')
            .groupBy('date').count().orderBy('date').toPandas())

if len(coverage):
    day_min, day_max = coverage['date'].min(), coverage['date'].max()
    expected = set(pd.date_range(day_min, day_max).date)
    actual = set(coverage['date'])
    missing = sorted(expected - actual)
    print(f'coverage: {day_min} .. {day_max}  ({len(actual):,} days)')
    if missing:
        print(f'MISSING {len(missing)} days: {missing[:20]}{" ..." if len(missing) > 20 else ""}')
    else:
        print('no gaps')
else:
    print('fact_employee_schedules is empty - nothing ingested yet')

## Rows per day (sales) – quick volume trend

In [ ]:
display(spark.table(f'{CATALOG}.fact.fact_sales_transactions')
        .groupBy('transaction_date').count().orderBy('transaction_date'))